[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Files, Paths and Formats](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)

# CSV


## What you will be able to do

Read and write CSV files correctly, including the rows that contain commas, quotes and line
breaks inside a single field. You will also be able to say why splitting on commas is wrong,
and what to do about rows that are not the length you expected.


## The idea

### The problem

CSV looks like the simplest format there is. Values separated by commas, one row per line. It
takes about four minutes to write a reader for it, and that reader will be wrong.

It is wrong because fields can contain commas. And quotes. And line breaks. A file of names is
fine until someone is called `Smith, John`, and then a program that splits on commas produces
one extra column for that row and none for the others, and nothing reports a problem.

There is a specification, it handles all of this, and Python implements it. The work is
already done.

### What CSV actually is

> A **CSV file** is text in which each line is a record and each record holds fields separated
> by a delimiter, usually a comma. A field containing the delimiter, a quote or a line break is
> **quoted**, and a quote inside a quoted field is written twice.
>
> The `csv` module reads and writes that format. `csv.reader` gives each row as a list;
> `csv.DictReader` gives each row as a dictionary keyed by the header.

The quoting rules are the part people do not implement, and the part that matters.

### What CSV does not have

Three absences, and each one causes a distinct problem later.

**No types.** Every value comes back as a string. `1`, `1.0` and `true` are all text, and it is
your job to convert them.

**No encoding.** A CSV file does not record whether it is UTF-8. Everything in the
**Encodings** notebook applies, including the byte order mark Excel writes.

**No agreed delimiter.** Comma is usual. Excel on a machine set to a European locale writes
semicolons, because in those locales the comma is the decimal separator. Tab-separated files are
common in bioinformatics and elsewhere.

### Where you will meet this

Constantly, and more than any other format. Exports from spreadsheets, databases, survey tools
and government portals all default to CSV.

The **Pandas** guide reads CSV in one line and handles most of this for you. It is worth knowing
what that line is doing, because when it goes wrong the fix is expressed in these terms.

### What this notebook covers

- Why `split(",")` fails, demonstrated on a file that breaks it
- `csv.reader` and `csv.DictReader`, and which to prefer
- Writing, and the quoting the writer adds without being asked
- `newline=""`, and the blank rows you get without it
- Rows that are too short or too long, and how to notice
- Delimiters other than the comma, and detecting one
- Three errors, plus the type mistake that raises nothing

### A first look

Nothing to run yet.

```python
line = 'Smith, John,engineer,plain'

print(line.split(","))
# ['Smith', ' John', 'engineer', 'plain']      four fields, and the name is broken

import csv
print(next(csv.reader(['"Smith, John",engineer,plain'])))
# ['Smith, John', 'engineer', 'plain']         three fields, and the name is intact
```

The difference is quoting, and it is the whole reason the `csv` module exists.


## Setup

Three imports and a folder to work in.

- `csv` reads and writes the format, and is the subject of this notebook
- `Path` builds paths and writes the example files
- `shutil` removes the scratch folder at the end

**Run this cell before the rest of the notebook.**


In [1]:
import csv
from pathlib import Path
import shutil

scratch = Path("scratch")
scratch.mkdir(exist_ok=True)

people = scratch / "people.csv"
people.write_text(
    'name,role,note\n'
    'Ada Lovelace,analyst,"likes commas, apparently"\n'
    '"Smith, John",engineer,plain\n'
    'Grace,"said ""hello""",quoted quote\n',
    encoding="utf-8",
)

print(people.read_text(encoding="utf-8"))


name,role,note
Ada Lovelace,analyst,"likes commas, apparently"
"Smith, John",engineer,plain
Grace,"said ""hello""",quoted quote



## Worked examples

### Why splitting on commas fails

That file has three columns. Splitting on commas does not agree.


In [2]:
for line in people.read_text(encoding="utf-8").splitlines()[1:]:
    fields = line.split(",")
    print(len(fields), fields)


4 ['Ada Lovelace', 'analyst', '"likes commas', ' apparently"']
4 ['"Smith', ' John"', 'engineer', 'plain']
3 ['Grace', '"said ""hello"""', 'quoted quote']


Four fields, four fields, three fields, from a three-column file. The first row was cut in the
middle of a note. The second lost the person's name to the split and kept the quote marks as
part of the data.

Nothing raised. A program doing this would carry on with `'"likes commas'` as a value.

### csv.reader gets it right


In [3]:
with open(people, newline="", encoding="utf-8") as f:
    for row in csv.reader(f):
        print(len(row), row)


3 ['name', 'role', 'note']
3 ['Ada Lovelace', 'analyst', 'likes commas, apparently']
3 ['Smith, John', 'engineer', 'plain']
3 ['Grace', 'said "hello"', 'quoted quote']


Three fields on every row. The quotes are gone from the values, because they were never data;
they were the format saying "the comma inside here is not a separator". And `said ""hello""`
came back as `said "hello"`, because a doubled quote inside a quoted field means one quote.

Note the two arguments to `open`. `newline=""` has a section of its own below. `encoding="utf-8"`
is there because the **Encodings** notebook argued it always should be.

### DictReader, which is usually the one you want

`csv.reader` gives you positions. `DictReader` uses the first row as a header and gives you
names.


In [4]:
with open(people, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        print(row["name"], "|", row["role"])


Ada Lovelace | analyst
Smith, John | engineer
Grace | said "hello"


`row["name"]` survives someone inserting a column. `row[0]` does not, and the failure is silent:
the code keeps running with the wrong field.

Each row is a dictionary, so everything from the **Dictionaries** notebook applies.


In [5]:
with open(people, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    print("columns:", reader.fieldnames)
    first = next(reader)

print(dict(first))


columns: ['name', 'role', 'note']
{'name': 'Ada Lovelace', 'role': 'analyst', 'note': 'likes commas, apparently'}


### Everything is text

This catches people every time.


In [6]:
numbers = scratch / "numbers.csv"
numbers.write_text("count,price\n3,4.50\n", encoding="utf-8")

with open(numbers, newline="", encoding="utf-8") as f:
    row = next(csv.DictReader(f))

print({key: (value, type(value).__name__) for key, value in row.items()})


{'count': ('3', 'str'), 'price': ('4.50', 'str')}


`'3'` and `'4.50'`, both strings. `row["count"] + 1` raises, and `row["count"] * 2` gives `'33'`,
which is worse because it does not raise.

Convert as you read, and do it explicitly:


In [7]:
with open(numbers, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        count = int(row["count"])
        price = float(row["price"])
        print(f"{count} x {price} = {count * price:.2f}")


3 x 4.5 = 13.50


Wrap that in `try` and `except ValueError` when the data is not yours, which the
**Errors and Exceptions** notebook covered. A single bad cell in a file of ten thousand rows
should not stop the run.


### Writing, and the quoting you get for free


In [8]:
out = scratch / "written.csv"

rows = [
    ["plain", "has, comma", 'has "quote"', "has\nnewline"],
]

with open(out, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows(rows)

print(repr(out.read_text(encoding="utf-8")))


'plain,"has, comma","has ""quote""","has\nnewline"\n'


The writer quoted the three fields that needed it and left the first alone. It doubled the
internal quote. It kept the line break inside the field, which is legal and is why a CSV file
cannot be counted by lines.

Reading it back gives exactly what went in.


In [9]:
with open(out, newline="", encoding="utf-8") as f:
    print(next(csv.reader(f)))


['plain', 'has, comma', 'has "quote"', 'has\nnewline']


`DictWriter` is the writing counterpart to `DictReader`, and it needs the field names up front.


In [10]:
out = scratch / "dictwritten.csv"

with open(out, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "score"])
    writer.writeheader()
    writer.writerow({"name": "Ada", "score": 91})
    writer.writerow({"name": "Smith, John", "score": 88})

print(out.read_text(encoding="utf-8"))


name,score
Ada,91
"Smith, John",88



### newline="", and the blank rows without it

The **Reading and Writing Text** notebook mentioned this and deferred the reason. Here it is.

The `csv` writer ends every row with `\r\n`, because that is what the format specifies. On
Windows, text mode **also** translates every `\n` into `\r\n`. Applied to what the writer
produced, that gives `\r\r\n`, and a blank row between every real one.

That does not happen on macOS or Linux, so it cannot be demonstrated directly here. The cell
below reproduces it by asking for the translation explicitly, which is exactly what Windows text
mode does by default.


In [11]:
demo_rows = [["a", "b"], ["1", "2"]]

windows_like = scratch / "windows_like.csv"
with open(windows_like, "w", encoding="utf-8", newline="\r\n") as f:
    csv.writer(f).writerows(demo_rows)

correct = scratch / "correct.csv"
with open(correct, "w", encoding="utf-8", newline="") as f:
    csv.writer(f).writerows(demo_rows)

print("without newline='' on Windows:", windows_like.read_bytes())
print("with newline='':              ", correct.read_bytes())


without newline='' on Windows: b'a,b\r\r\n1,2\r\r\n'
with newline='':               b'a,b\r\n1,2\r\n'


In [12]:
with open(windows_like, newline="", encoding="utf-8") as f:
    print("rows read back:", list(csv.reader(f)))


rows read back: [['a', 'b'], [], ['1', '2'], []]


An empty row after every real one. In Excel that is a blank line between each pair of records,
and it is the single most common complaint about CSV files written by Python.

**Pass `newline=""` to every `open` used with the `csv` module**, reading and writing. It costs
nothing on the systems where it makes no difference, and it prevents this on the one where it
does.


### Rows that are not the right length

Real files have ragged rows, and neither reader complains.


In [13]:
ragged = scratch / "ragged.csv"
ragged.write_text("a,b,c\n1,2,3\n4,5\n6,7,8,9\n", encoding="utf-8")

with open(ragged, newline="", encoding="utf-8") as f:
    for i, row in enumerate(csv.reader(f)):
        print(f"row {i}: {len(row)} fields {row}")


row 0: 3 fields ['a', 'b', 'c']
row 1: 3 fields ['1', '2', '3']
row 2: 2 fields ['4', '5']
row 3: 4 fields ['6', '7', '8', '9']


Two, three and four fields, with no error. `csv.reader` reports what is there.

`DictReader` handles it differently, and its defaults are worth seeing.


In [14]:
with open(ragged, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        print(dict(row))


{'a': '1', 'b': '2', 'c': '3'}
{'a': '4', 'b': '5', 'c': None}
{'a': '6', 'b': '7', 'c': '8', None: ['9']}


A short row gets `None` for the missing column. A long row puts the surplus under the key `None`,
which is an awkward thing to find in your data.

Name them and the result reads better:


In [15]:
with open(ragged, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f, restkey="extra", restval="MISSING"):
        print(dict(row))


{'a': '1', 'b': '2', 'c': '3'}
{'a': '4', 'b': '5', 'c': 'MISSING'}
{'a': '6', 'b': '7', 'c': '8', 'extra': ['9']}


Better still, check as you go. A count of bad rows at the end is worth more than a clean run
that quietly dropped data, which is the lesson from the **Debugging** notebook applied to files.


In [16]:
expected = 3
good = bad = 0

with open(ragged, newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    header = next(reader)
    for i, row in enumerate(reader, start=2):
        if len(row) == expected:
            good += 1
        else:
            bad += 1
            print(f"  line {i}: expected {expected} fields, found {len(row)}")

print(f"{good} good rows, {bad} bad")


  line 3: expected 3 fields, found 2
  line 4: expected 3 fields, found 4
1 good rows, 2 bad


### Delimiters other than the comma


In [17]:
semi = scratch / "european.csv"
semi.write_text("name;value\nada;1\n", encoding="utf-8")

with open(semi, newline="", encoding="utf-8") as f:
    print("read as comma-separated:", list(csv.reader(f)))

with open(semi, newline="", encoding="utf-8") as f:
    print("with delimiter=';':     ", list(csv.reader(f, delimiter=";")))


read as comma-separated: [['name;value'], ['ada;1']]
with delimiter=';':      [['name', 'value'], ['ada', '1']]


The first version did not fail. It produced one column per row whose value happens to contain
semicolons, which is a correct reading of a file that is not what you thought.

`csv.Sniffer` guesses from a sample when you do not know.


In [18]:
sample = semi.read_text(encoding="utf-8")

dialect = csv.Sniffer().sniff(sample)

print("detected delimiter:", repr(dialect.delimiter))
print("looks like it has a header:", csv.Sniffer().has_header(sample))


detected delimiter: ';'
looks like it has a header: True


`Sniffer` is a guess and it can be wrong, particularly on small samples or files with unusual
quoting. Use it to investigate an unfamiliar file, and hardcode the delimiter once you know it.


## Your turn

Six tasks. Write your answer in the cell under each and run it.

Try each one before you look at an answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/04-csv-solutions.ipynb).

**1.** Write a CSV to `scratch/cities.csv` with the header `city,country,population` and three
rows, one of which has a comma inside the city name. Print the raw file text.


In [19]:
# your code here


**2.** Read it back with `csv.reader` and print the number of fields in each row, to show the
comma did not break it.


In [20]:
# your code here


**3.** Read it with `DictReader` and print each city with its country.


In [21]:
# your code here


**4.** Total the populations. Remember what type they come back as.


In [22]:
# your code here


**5.** Write a semicolon-separated version of the same data, then read it back correctly.


In [23]:
# your code here


**6.** Write a file with one row that has too few fields, read it with `DictReader` using
`restval`, and print the result.


In [24]:
# your code here


## Common errors

Each cell below is run on purpose so you can see the real message.

### The quiet one: doing arithmetic on text


In [25]:
prices = scratch / "prices.csv"
prices.write_text("item,price\napple,3\npear,4\n", encoding="utf-8")

total = ""
with open(prices, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        total += row["price"]

print("total:", repr(total))


total: '34'


`'34'`. The prices were joined as text rather than added as numbers, and nothing failed.

`+` works on strings, which is why this is silent. Had the code used `-` it would have raised
immediately and been fixed in a minute.

Convert on the way in, and the problem cannot arise:


In [26]:
total = 0
with open(prices, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        total += int(row["price"])

print("total:", total)


total: 7


### KeyError: the column is not called what you think


In [27]:
with open(prices, newline="", encoding="utf-8") as f:
    row = next(csv.DictReader(f))

print(row["Price"])


KeyError: 'Price'

`KeyError: 'Price'`. Column names are case sensitive and often carry stray spaces from whatever
produced the file.

`reader.fieldnames` shows exactly what they are, with `repr` so the spaces are visible:


In [28]:
with open(prices, newline="", encoding="utf-8") as f:
    print([repr(name) for name in csv.DictReader(f).fieldnames])


["'item'", "'price'"]


When a column you can see in Excel raises a `KeyError`, this is the first thing to check, along
with the byte order mark from the **Encodings** notebook.


### ValueError: the file was closed before the rows were read

`csv.reader` reads lazily, taking rows as you ask for them.


In [29]:
with open(prices, newline="", encoding="utf-8") as f:
    reader = csv.reader(f)

print(list(reader))


ValueError: I/O operation on closed file.

`I/O operation on closed file`. The reader was created inside the block and used outside it, and
by then the file was gone.

Do the reading inside the block. If you need the rows afterward, build the list inside:


In [30]:
with open(prices, newline="", encoding="utf-8") as f:
    rows = list(csv.reader(f))

print(rows)


[['item', 'price'], ['apple', '3'], ['pear', '4']]


### TypeError: writing a row that is not a sequence


In [31]:
with open(scratch / "bad.csv", "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerow("abc")
    csv.writer(f).writerows(["abc", "def"])

print(repr((scratch / "bad.csv").read_text(encoding="utf-8")))


'a,b,c\na,b,c\nd,e,f\n'


No error, and almost certainly not what was wanted. A string is a sequence of characters, so
`writerow("abc")` wrote three fields: `a`, `b`, `c`.

`writerow` takes a list of fields and `writerows` takes a list of those. Passing a bare string
to either is legal and wrong.


In [32]:
with open(scratch / "good.csv", "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerow(["abc"])

print(repr((scratch / "good.csv").read_text(encoding="utf-8")))


'abc\n'


### Cleaning up


In [33]:
shutil.rmtree(scratch)

print("scratch still there:", scratch.exists())


scratch still there: False


## Recap

- Never split a CSV line on commas. Fields can contain commas, quotes and line breaks.
- `csv.reader` gives lists; `csv.DictReader` gives dictionaries keyed by the header, and is
  usually the one to use.
- Every value arrives as a **string**. Convert explicitly, and expect bad values.
- Pass `newline=""` to every `open` used with `csv`, or Windows adds a blank row after each one.
- Also pass `encoding=`, and `utf-8-sig` for anything that came from Excel.
- Ragged rows raise nothing: `DictReader` fills with `None` and collects extras under `None`.
  Name them with `restval` and `restkey`, and count the bad rows.
- The delimiter is not always a comma; `csv.Sniffer` guesses and can be wrong.
- `writerow("abc")` writes three fields. It wants a list.


## What is next

The **JSON on Disk** notebook, which handles the other format data arrives in. JSON has the
types CSV lacks, which solves one problem and introduces others: what happens to a tuple, a
date, or a dictionary with integer keys when it makes the round trip.


---

&#8592; **Previous:** [Encodings](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/03-encodings.ipynb)  &nbsp;·&nbsp;  [Files, Paths and Formats Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)
